# Inventory Decision Engine
This notebook imports reusable project code from `src/` and uses the real local M5 data when available.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks": ROOT = ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))


In [ ]:
import pandas as pd
from src.config import settings
from src.data.loader import load_m5_data
from src.data.preprocessing import prepare_m5_long
prepared_path = settings.processed_dir / "m5_prepared.csv"
if prepared_path.exists():
    prepared = pd.read_csv(prepared_path, parse_dates=["date"])
else:
    data = load_m5_data()
    prepared = prepare_m5_long(data["sales"], data["calendar"], data["prices"])
print(prepared.shape)


In [ ]:
from src.forecasting.trainer import train_with_holdout
result = train_with_holdout(prepared, 28)


In [ ]:
from src.forecasting.uncertainty import add_prediction_intervals
validation = result["validation"]
intervals = add_prediction_intervals(validation, validation["residual"], 0.90)
intervals.head()

In [ ]:
from src.inventory.decision_engine import inventory_decision
example = intervals.sort_values("date").head(28)
inventory_decision(example["forecast"].tolist(), example["lower_bound"].tolist(), example["upper_bound"].tolist(), current_inventory=50)